# EEG_08b — TemporalChebGCN Subject-Specific (Leave-One-Session-Out)

## Cosa fa questo notebook

Addestra un modello **ChebGCN con Temporal Encoder** in modalità **subject-specific**:
per ogni soggetto viene addestrato un modello dedicato, usando leave-one-session-out.

## Differenza con EEG_08
- **EEG_08** usa split subject-independent (train sogg 0-49, val 50-59, test 60-73),
  un unico modello addestrato su tutti i soggetti, grafo PCC calcolato sul training globale.
- **EEG_08b** usa split subject-specific (leave-one-session-out per ogni soggetto),
  un modello per soggetto, grafo PCC calcolato sui dati di training del singolo soggetto.

## Differenza con EEG_06
- **EEG_06** usa modelli Braindecode (EEGNet, ShallowFBCSPNet, Deep4Net, EEGConformer, ATCNet, Labram)
  che processano il segnale EEG grezzo come tensore `(batch, 59, 384)`.
- **EEG_08b** usa modelli PyTorch Geometric (ChebGCN_2L, ChebGCN_3L, ChebGCNSkip) che processano
  il segnale EEG come grafi `Data(x=(59, 384), edge_index, y)` con grafo PCC per-soggetto.

## Split
- Train: sessioni 1-3
- Val: sessione 4
- Test: sessione 5

## Architettura
```
Input: grafo per-soggetto (PCC k-NN calcolato sui dati training del soggetto)
  Nodi V = 59 elettrodi EEG
  Feature nodo = serie temporale raw (384 campioni, normalizzata)

[Temporal Encoder] Conv1d(1→32, k=25) → BN → ELU → Conv1d(32→64, k=25) → BN → ELU → AdaptiveAvgPool1d(1)
[ChebConv 1]       ChebConv(64 → 64, K=2) + ELU
[ChebConv 2]       ChebConv(64 → 32, K=2) + ELU
[Global MeanPool]  (batch, 32)
[Linear]           Linear(32 → n_classes)
```

## Riferimento grafo+conv
Lun X et al., *GCNs-Net*, IEEE TNSRE 2022, arXiv:2006.08924

Data: 2026-04-18

In [ ]:
# ═══════════════════════════════════════════════════════════
#  TOGGLE CONFIG — modifica qui per cambiare parametri
# ═══════════════════════════════════════════════════════════
CLUSTER_SCHEME    = "concr4"   # "concr4" | "phon4" | "sem5" | "pos4" | "ward5" | "ward4"
USE_CLUSTERS      = True        # False → 110 parole originali
USE_INSTANCE_NORM = True        # Bomatter et al. 2024: normalizza ogni trial indipendentemente

# ── Configurazione grafo ────────────────────────────────────
K_GRAPH = 6     # k-NN sul PCC: ogni elettrodo connesso ai k più correlati

# ── Configurazione soggetti e sessioni ──────────────────────
N_SUBJECTS_TEST = 10   # None → tutti i 70 soggetti | int → primi N soggetti (test veloce)
SESSION_TEST    = 5    # sessione usata come test set (1-5)
SESSIONS_TRAIN  = [1, 2, 3, 4]  # sessioni usate come train+val

# ── Sweep config ────────────────────────────────────────────
SWEEP_RESUME = True   # se True, salta soggetti già calcolati nel CSV intermedio

# ── Training ────────────────────────────────────────────────
MAX_EPOCHS   = 100
PATIENCE     = 15
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 32

SEED = 42
# ═══════════════════════════════════════════════════════════

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP su macOS

import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

# PyTorch Geometric
from torch_geometric.data import Data, Dataset as PyGDataset
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import ChebConv, global_mean_pool

# Device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

print("device:", device)
print("Python:", __import__('sys').version.split()[0])
print("torch:", torch.__version__)
import torch_geometric; print("torch_geometric:", torch_geometric.__version__)

In [ ]:
# ============================================================
# CONFIGURAZIONE PERCORSI + CARICAMENTO METADATA
# ============================================================

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

META_CSV    = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH   = project_root / "src" / "io" / "ebneuro.locs"
interim_dir = project_root / "data" / "interim"

# Parametri EEG
N_CHANS = 59    # canali dopo rimozione A1, A2
N_TIMES = 384   # campioni a 256 Hz (~1.5s)
SFREQ   = 256

print(f"project_root: {project_root}")

# ── Caricamento metadata ────────────────────────────────────
import sys
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

meta = pd.read_csv(META_CSV)

# Filtro righe corrotte (epoch_idx fuori range in file H5 noti)
_initial_len = len(meta)
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34))  |
    (meta["path_h5"].str.contains("ignore_8_05.h5") & (meta["epoch_idx"] >= 110))
)]
if len(meta) < _initial_len:
    print(f"Rimosse {_initial_len - len(meta)} righe corrotte dai metadati.")

meta["subject_id"] = meta["subject_id"].astype(str).str.zfill(2)
meta = meta[pd.to_numeric(meta["subject_id"], errors="coerce").notna()]  # rimuove ignore_43

# ── Indici canali: rimuovi A1 (idx=0) e A2 (idx=7) ──────────
def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]  # H5 ha 61 canali registrati

ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE  = {"A1", "A2"}
keep_idx = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]
assert len(keep_idx) == N_CHANS, f"Attesi {N_CHANS} canali, trovati {len(keep_idx)}"

# ── Schema label ────────────────────────────────────────────
_scheme = CLUSTER_SCHEME if USE_CLUSTERS else "raw110"
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(_scheme, interim_dir)

chance_level = 1.0 / N_CLASSES
NORM_TAG     = "_norm" if USE_INSTANCE_NORM else ""

print(f"Meta: {len(meta)} epoche | {meta['subject_id'].nunique()} soggetti")
print(f"Canali: {len(keep_idx)} ({keep_names[:4]}...)")
print(f"Schema: {_scheme} | {N_CLASSES} classi | Chance level: {100/N_CLASSES:.1f}%")
for cid, cname in cluster_names.items():
    n = sum(1 for v in labelid2cluster.values() if v == cid)
    print(f"  {cid} — {cname}: {n} parole")

In [ ]:
# ============================================================
# FUNZIONE build_pcc_graph — grafo PCC per soggetto
# ============================================================

def build_pcc_graph(x_train, k=6):
    """
    Calcola il grafo PCC k-NN dai dati di training di un singolo soggetto.

    Args:
        x_train : np.array (N_trials, 59, T) — dati di training del soggetto
        k       : int — numero di vicini k-NN per elettrodo

    Returns:
        edge_index : torch.LongTensor (2, E) — indici degli archi del grafo

    Design:
        - Calcola la matrice PCC media su tutti i trial di training
        - Reshape a (N*T, C) per corrcoef veloce su tutta la sequenza
        - k-NN bidirezionale: per ogni nodo i, aggiunge archi ai top-k nodi più correlati
        - Rimuove duplicati con torch.unique
    """
    N_trials, C, T = x_train.shape
    # reshape a (N*T, C) per corrcoef veloce
    flat = x_train.transpose(0, 2, 1).reshape(-1, C)  # (N*T, C)
    corr = np.corrcoef(flat.T)  # (C, C)
    np.fill_diagonal(corr, 0)
    src, dst = [], []
    for i in range(C):
        top_k = np.argsort(corr[i])[-k:]
        for j in top_k:
            src.append(i); dst.append(j)
            src.append(j); dst.append(i)
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    # rimuovi duplicati
    edge_index = torch.unique(edge_index, dim=1)
    return edge_index


print("build_pcc_graph OK")

In [ ]:
# ============================================================
# DATASET PyG — SubjectPyGDataset
# ============================================================

class SubjectPyGDataset(PyGDataset):
    """
    Dataset PyG per la modalità subject-specific.
    Ogni item: Data(x=(59, 384), edge_index, y=cluster_label).

    Caricamento lazy da file H5 per efficienza memoria.
    Normalizzazione per-canale con mean/std del training set (passati come parametri).
    Instance norm applicata DOPO la normalizzazione globale (Bomatter et al. 2024).

    Nota metodologica:
    - edge_index è lo stesso per tutti i trial del soggetto (calcolato una volta sul train)
    - mean/std calcolati SOLO sul training set, passati a val/test via costruttore
    - file_cache usa apertura lazy per minimizzare overhead I/O
    """
    def __init__(self, records, keep_idx, labelid2cluster, edge_index,
                 mean=None, std=None, instance_norm=False, transform=None):
        """
        Args:
            records       : lista di dict con chiavi path_h5, epoch_idx, label_idx
            keep_idx      : list[int] — indici canali da tenere (es. esclude A1, A2)
            labelid2cluster: dict label_idx -> cluster_id
            edge_index    : torch.LongTensor (2, E) — grafo del soggetto (PCC k-NN)
            mean          : torch.FloatTensor (59, 1) — se None, calcolato sul dataset
            std           : torch.FloatTensor (59, 1) — se None, calcolato sul dataset
            instance_norm : bool — se True, applica instance norm per trial
        """
        super().__init__(root=None, transform=transform)
        self.records         = records
        self.keep_idx        = keep_idx
        self.labelid2cluster = labelid2cluster
        self.edge_index      = edge_index
        self.instance_norm   = instance_norm
        self.file_cache      = {}
        self.mean = mean
        self.std  = std
        if mean is None:
            self._compute_stats()

    def _compute_stats(self, seed=42):
        """Calcola mean/std per-canale su un campione del dataset."""
        rng = np.random.RandomState(seed)
        n   = min(500, len(self.records))
        idxs = rng.choice(len(self.records), n, replace=False)
        paths_map = defaultdict(list)
        for idx in idxs:
            r = self.records[idx]
            paths_map[r["path_h5"]].append(int(r["epoch_idx"]))
        buf = []
        for path, epoch_idxs in paths_map.items():
            with h5py.File(path, "r") as f:
                for e_idx in epoch_idxs:
                    buf.append(f["data"][e_idx][self.keep_idx, :].astype(np.float32))
        buf = np.stack(buf)  # (N, 59, T)
        self.mean = torch.tensor(
            buf.mean(axis=(0, 2), keepdims=False).reshape(-1, 1), dtype=torch.float32
        )
        self.std = torch.tensor(
            buf.std(axis=(0, 2), keepdims=False).reshape(-1, 1).clip(1e-6), dtype=torch.float32
        )

    def __del__(self):
        for f in self.file_cache.values():
            try:
                f.close()
            except Exception:
                pass

    def len(self):
        return len(self.records)

    def get(self, idx):
        r    = self.records[idx]
        path = r["path_h5"]
        if path not in self.file_cache:
            self.file_cache[path] = h5py.File(path, "r")
        x_np = self.file_cache[path]["data"][int(r["epoch_idx"])][self.keep_idx, :].astype(np.float32)
        x    = torch.tensor(x_np, dtype=torch.float32)
        # normalizzazione per-canale (statistiche del training set)
        x = (x - self.mean) / self.std
        # instance norm per trial (Bomatter et al. 2024: rimuove bias per-soggetto)
        if self.instance_norm:
            x = (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-6)
        label = self.labelid2cluster[int(r["label_idx"])]
        return Data(
            x          = x,                                      # (59, 384)
            edge_index = self.edge_index,                        # (2, E)
            y          = torch.tensor(label, dtype=torch.long),
        )


print("SubjectPyGDataset OK")

In [ ]:
# ============================================================
# FUNZIONE make_subject_splits
# ============================================================

def make_subject_splits(meta_df, keep_idx, labelid2cluster, subject_id,
                         sessions_train, session_test, k=K_GRAPH, instance_norm=False):
    """
    Crea train/val/test SubjectPyGDataset per un singolo soggetto.

    Split:
        Train: sessions_train esclusa l'ultima (es. [1,2,3])
        Val:   ultima sessione del blocco train (es. sessione 4)
        Test:  session_test (es. sessione 5)

    Grafo:
        edge_index calcolato dai dati di training del soggetto tramite build_pcc_graph.
        Lo stesso grafo viene condiviso tra train, val e test del soggetto.

    Returns:
        (ds_train, ds_val, ds_test, edge_index) oppure ([], [], [], None) se dati insufficienti
    """
    subj_str   = str(int(subject_id)).zfill(2) if not isinstance(subject_id, str) else subject_id
    meta_subj  = meta_df[meta_df["subject_id"] == subj_str].copy()

    meta_trainval = meta_subj[meta_subj["session_id"].isin(sessions_train)]
    meta_test     = meta_subj[meta_subj["session_id"] == session_test]

    last_train_sess = max(sessions_train)
    meta_train = meta_trainval[meta_trainval["session_id"] != last_train_sess]
    meta_val   = meta_trainval[meta_trainval["session_id"] == last_train_sess]

    r_tr = meta_train[["path_h5", "epoch_idx", "label_idx"]].to_dict("records")
    r_va = meta_val  [["path_h5", "epoch_idx", "label_idx"]].to_dict("records")
    r_te = meta_test [["path_h5", "epoch_idx", "label_idx"]].to_dict("records")

    if len(r_tr) == 0 or len(r_va) == 0 or len(r_te) == 0:
        return [], [], [], None

    # Calcola grafo PCC dai dati di training del soggetto
    # Carica in memoria i trial di training per il calcolo PCC
    x_train_list = []
    paths_map    = defaultdict(list)
    for r in r_tr:
        paths_map[r["path_h5"]].append(int(r["epoch_idx"]))
    for path, epoch_idxs in paths_map.items():
        with h5py.File(path, "r") as f:
            for e_idx in epoch_idxs:
                x_train_list.append(f["data"][e_idx][keep_idx, :].astype(np.float32))
    x_train_np = np.stack(x_train_list)  # (N_trials_train, 59, T)

    edge_index = build_pcc_graph(x_train_np, k=k)

    ds_train = SubjectPyGDataset(
        r_tr, keep_idx, labelid2cluster, edge_index, instance_norm=instance_norm
    )
    ds_val = SubjectPyGDataset(
        r_va, keep_idx, labelid2cluster, edge_index,
        mean=ds_train.mean, std=ds_train.std, instance_norm=instance_norm
    )
    ds_test = SubjectPyGDataset(
        r_te, keep_idx, labelid2cluster, edge_index,
        mean=ds_train.mean, std=ds_train.std, instance_norm=instance_norm
    )

    return ds_train, ds_val, ds_test, edge_index


print("make_subject_splits OK")

In [ ]:
# ============================================================
# MODELLI — TemporalEncoder + ChebGCN varianti
# Riferimento: Lun et al. 2022 (GCNs-Net, IEEE TNSRE, arXiv:2006.08924)
# ============================================================

class TemporalEncoder(nn.Module):
    """
    1D CNN per-nodo (pesi condivisi tra tutti gli elettrodi).
    Estrae embedding compatto dalla serie temporale di ogni canale EEG.

    Input:  (N_nodes_total, N_TIMES) — tutti i nodi del batch concatenati
    Output: (N_nodes_total, 64)      — embedding per nodo
    """
    def __init__(self, n_times: int = N_TIMES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=25, padding=12),
            nn.BatchNorm1d(32),
            nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=25, padding=12),
            nn.BatchNorm1d(64),
            nn.ELU(),
            nn.AdaptiveAvgPool1d(1),  # comprime T→1
            nn.Flatten(),             # (N_nodes, 64)
        )

    def forward(self, x):
        # x: (N_nodes_total, N_TIMES)
        return self.net(x.unsqueeze(1))  # (N_nodes_total, 64)


class ChebGCN_2L(nn.Module):
    """
    TemporalEncoder → 2x ChebConv(K=2) → global_mean_pool → Linear
    Architettura base da Lun et al. 2022, adattata per trial interi.
    """
    def __init__(self, n_classes=4, dropout=0.5):
        super().__init__()
        self.temporal  = TemporalEncoder(N_TIMES)
        self.conv1     = ChebConv(64, 64, K=2)
        self.conv2     = ChebConv(64, 32, K=2)
        self.drop      = nn.Dropout(dropout)
        self.bn1       = nn.BatchNorm1d(64)
        self.bn2       = nn.BatchNorm1d(32)
        self.classifier = nn.Linear(32, n_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.temporal(x)                              # (N, 64)
        x = F.elu(self.bn1(self.conv1(x, edge_index)))    # ChebConv 1
        x = self.drop(x)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))    # ChebConv 2
        x = global_mean_pool(x, batch)                    # (batch, 32)
        return self.classifier(x)


class ChebGCN_3L(nn.Module):
    """
    TemporalEncoder → 3x ChebConv(K=2) → global_mean_pool → Linear
    Variante con layer aggiuntivo per ricettività spettrale maggiore.
    """
    def __init__(self, n_classes=4, dropout=0.5):
        super().__init__()
        self.temporal  = TemporalEncoder(N_TIMES)
        self.conv1     = ChebConv(64, 64, K=2)
        self.conv2     = ChebConv(64, 64, K=2)
        self.conv3     = ChebConv(64, 32, K=2)
        self.drop      = nn.Dropout(dropout)
        self.bn1       = nn.BatchNorm1d(64)
        self.bn2       = nn.BatchNorm1d(64)
        self.bn3       = nn.BatchNorm1d(32)
        self.classifier = nn.Linear(32, n_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.temporal(x)
        x = F.elu(self.bn1(self.conv1(x, edge_index))); x = self.drop(x)
        x = F.elu(self.bn2(self.conv2(x, edge_index))); x = self.drop(x)
        x = F.elu(self.bn3(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.classifier(x)


class ChebGCNSkip(nn.Module):
    """
    TemporalEncoder → ChebConv + skip connection → global_mean_pool → Linear
    La skip connection (residuale) stabilizza il training su dataset piccoli.
    """
    def __init__(self, n_classes=4, dropout=0.5):
        super().__init__()
        self.temporal  = TemporalEncoder(N_TIMES)
        self.proj      = nn.Linear(64, 64)  # proiezione skip
        self.conv1     = ChebConv(64, 64, K=2)
        self.conv2     = ChebConv(64, 32, K=2)
        self.drop      = nn.Dropout(dropout)
        self.bn1       = nn.BatchNorm1d(64)
        self.bn2       = nn.BatchNorm1d(32)
        self.classifier = nn.Linear(32, n_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h    = self.temporal(x)                              # (N, 64)
        skip = self.proj(h)                                  # skip (N, 64)
        h    = F.elu(self.bn1(self.conv1(h, edge_index) + skip))  # ChebConv 1 + skip
        h    = self.drop(h)
        h    = F.elu(self.bn2(self.conv2(h, edge_index)))    # ChebConv 2
        h    = global_mean_pool(h, batch)
        return self.classifier(h)


# Factory
MODEL_FACTORIES = {
    "ChebGCN_2L":  lambda nc: ChebGCN_2L(n_classes=nc),
    "ChebGCN_3L":  lambda nc: ChebGCN_3L(n_classes=nc),
    "ChebGCNSkip": lambda nc: ChebGCNSkip(n_classes=nc),
}

def build_model(name, n_classes):
    return MODEL_FACTORIES[name](n_classes)


# Stampa conteggio parametri
print(f"{'Modello':<18} {'Parametri':>12}")
print("-" * 32)
for name, factory in MODEL_FACTORIES.items():
    m = factory(4)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:<18} {n_params:>12,}")

In [ ]:
# ============================================================
# FUNZIONI TRAINING E VALUTAZIONE
# ============================================================

def train_model(model, ds_train, ds_val, save_path, tb_dir,
                n_epochs=MAX_EPOCHS, patience=PATIENCE,
                lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE):
    """
    Training con early stopping su val_bacc e class weights anti-collapse.

    Args:
        model     : modello PyG da addestrare
        ds_train  : SubjectPyGDataset training set
        ds_val    : SubjectPyGDataset validation set
        save_path : Path — percorso checkpoint
        tb_dir    : Path — directory TensorBoard

    Returns:
        dict con val_acc, val_bacc, epochs, model
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    # Class weights anti-collapse (inversamente proporzionali alla frequenza)
    all_labels = [ds_train.labelid2cluster[int(r["label_idx"])] for r in ds_train.records]
    counts     = torch.bincount(torch.tensor(all_labels), minlength=N_CLASSES).float()
    class_weights = (1.0 / counts.clamp(min=1))
    class_weights = (class_weights / class_weights.sum() * N_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    writer    = SummaryWriter(log_dir=str(tb_dir))
    loader_tr = PyGDataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=0)
    loader_va = PyGDataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=0)

    best_val_bacc = -1.0
    best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_cnt  = 0
    history       = defaultdict(list)

    for epoch in range(n_epochs):
        # ── Fase di training ──
        model.train()
        loss_sum, correct, n_tot = 0.0, 0, 0
        pbar = tqdm(loader_tr, desc=f"Epoch {epoch+1}/{n_epochs}", leave=False)
        for data in pbar:
            data = data.to(device)
            optimizer.zero_grad()
            logits = model(data)
            loss   = criterion(logits, data.y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * len(data.y)
            correct  += (logits.argmax(1) == data.y).sum().item()
            n_tot    += len(data.y)
            pbar.set_postfix(loss=f"{loss.item():.3f}")
        scheduler.step()

        # ── Fase di validazione ──
        model.eval()
        ys_v, ps_v = [], []
        with torch.no_grad():
            for data in loader_va:
                data = data.to(device)
                ps_v.extend(model(data).argmax(1).cpu().tolist())
                ys_v.extend(data.y.cpu().tolist())

        val_acc    = accuracy_score(ys_v, ps_v)
        val_bacc   = balanced_accuracy_score(ys_v, ps_v)
        train_acc  = correct / n_tot if n_tot > 0 else 0.0
        train_loss = loss_sum / n_tot if n_tot > 0 else 0.0

        history["val_acc"].append(val_acc)
        history["val_bacc"].append(val_bacc)
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)

        writer.add_scalar("val/acc",    val_acc,   epoch)
        writer.add_scalar("val/bacc",   val_bacc,  epoch)
        writer.add_scalar("train/acc",  train_acc, epoch)
        writer.add_scalar("train/loss", train_loss, epoch)

        # ── Early stopping su val_bacc ──
        if val_bacc > best_val_bacc:
            best_val_bacc = val_bacc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
            save_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(best_state, save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                break

    writer.close()
    model.load_state_dict(best_state)
    torch.save(best_state, save_path)

    return {
        "val_acc":  max(history["val_acc"]) if history["val_acc"] else 0.0,
        "val_bacc": best_val_bacc,
        "epochs":   epoch + 1,
        "model":    model,
    }


def evaluate(model, ds, batch_size=BATCH_SIZE):
    """Calcola acc e bacc su un SubjectPyGDataset."""
    loader = PyGDataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval().to(device)
    ys, ps = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            ps.extend(model(data).argmax(1).cpu().tolist())
            ys.extend(data.y.cpu().tolist())
    return {
        "acc":    accuracy_score(ys, ps),
        "bacc":   balanced_accuracy_score(ys, ps),
        "y_true": np.array(ys),
        "y_pred": np.array(ps),
    }


print("Funzioni training OK")

In [ ]:
# ============================================================
# LOOP PRINCIPALE — tutti i soggetti × tutti i modelli
# ============================================================

all_subjects = sorted(meta["subject_id"].unique())
if N_SUBJECTS_TEST:
    all_subjects = all_subjects[:N_SUBJECTS_TEST]

TB_BASE   = project_root / "runs"   / f"eeg08b_ss_{N_CLASSES}{NORM_TAG}"
CKPT_BASE = project_root / "models" / "eeg08b"
CKPT_BASE.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = project_root / "data" / "interim" / f"eeg08b_ss_{N_CLASSES}{NORM_TAG}_results.csv"

# ── Resume da CSV intermedio ─────────────────────────────────
if SWEEP_RESUME and RESULTS_CSV.exists():
    df_existing = pd.read_csv(RESULTS_CSV)
    all_results = df_existing.to_dict("records")
    done_pairs  = set(zip(df_existing["subject"].astype(str), df_existing["model"].astype(str)))
    print(f"Resume: trovati {len(all_results)} risultati già calcolati in {RESULTS_CSV.name}")
else:
    all_results = []
    done_pairs  = set()

MODEL_NAMES = list(MODEL_FACTORIES.keys())

# ── Stato rapido prima di partire ───────────────────────────
if done_pairs:
    _done_subj = sorted({s for s, _ in done_pairs})
    _todo_subj = [s for s in sorted(all_subjects) if s not in _done_subj]
    _n_models  = len(MODEL_NAMES)
    _complete  = [s for s in _done_subj
                  if sum(1 for m in MODEL_NAMES if (s, m) in done_pairs) == _n_models]
    _partial   = [s for s in _done_subj if s not in _complete]
    print(f"Soggetti completi ({_n_models}/{_n_models} modelli): {len(_complete)} → {_complete}")
    print(f"Soggetti parziali: {_partial}")
    print(f"Soggetti da fare:  {len(_todo_subj)} → {_todo_subj}")
print()

# ── Loop soggetti ────────────────────────────────────────────
for subj in tqdm(all_subjects, desc="Subjects"):
    # Controlla se il soggetto è completamente già fatto
    if SWEEP_RESUME and all((str(subj), m) in done_pairs for m in MODEL_NAMES):
        existing_bacc = [
            r["test_bacc"] for r in all_results
            if str(r["subject"]) == str(subj)
        ]
        print(f"Soggetto {subj} [SKIP — tutti i modelli nel CSV] | "
              f"mean_test_bacc={np.mean(existing_bacc):.4f}")
        continue

    # Carica splits (include calcolo grafo PCC del soggetto)
    ds_tr, ds_va, ds_te, edge_index = make_subject_splits(
        meta, keep_idx, labelid2cluster, subj,
        SESSIONS_TRAIN, SESSION_TEST,
        k=K_GRAPH, instance_norm=USE_INSTANCE_NORM
    )

    if len(ds_tr) == 0:
        print(f"Soggetto {subj}: dati insufficienti, skip")
        continue

    print(f"\n── Soggetto {subj} │ tr={len(ds_tr)} va={len(ds_va)} te={len(ds_te)} "
          f"| grafo: {edge_index.shape[1]} archi ──")

    for model_name in MODEL_NAMES:
        # ── Resume: skip se coppia (soggetto, modello) già nel CSV ──
        if SWEEP_RESUME and (str(subj), str(model_name)) in done_pairs:
            existing = next(
                r for r in all_results
                if str(r["subject"]) == str(subj) and r["model"] == model_name
            )
            print(f"  {model_name} [SKIP — già nel CSV]: test_bacc={existing['test_bacc']:.4f}")
            continue

        tb_dir    = TB_BASE / subj / model_name
        ckpt_path = CKPT_BASE / f"{subj}_{model_name}.pt"

        if SWEEP_RESUME and ckpt_path.exists():
            # Checkpoint esiste ma non nel CSV → valuta e aggiungi
            m_loaded = build_model(model_name, N_CLASSES)
            m_loaded.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
            va_r = evaluate(m_loaded, ds_va)
            te_r = evaluate(m_loaded, ds_te)
            row  = {
                "subject": subj, "model": model_name,
                "val_acc": va_r["acc"], "val_bacc": va_r["bacc"],
                "test_acc": te_r["acc"], "test_bacc": te_r["bacc"],
                "epochs": -1, "time_s": 0.0, "k_graph": K_GRAPH, "n_classes": N_CLASSES,
            }
            print(f"  {model_name} [CKPT→CSV]: test_bacc={te_r['bacc']:.4f}")
        else:
            model = build_model(model_name, N_CLASSES)
            t0    = time.time()
            res   = train_model(model, ds_tr, ds_va, ckpt_path, tb_dir)
            elapsed = time.time() - t0
            te_r  = evaluate(res["model"], ds_te)
            row   = {
                "subject": subj, "model": model_name,
                "val_acc": res["val_acc"], "val_bacc": res["val_bacc"],
                "test_acc": te_r["acc"], "test_bacc": te_r["bacc"],
                "epochs": res["epochs"], "time_s": round(elapsed, 1),
                "k_graph": K_GRAPH, "n_classes": N_CLASSES,
            }
            print(f"  {model_name}: test_bacc={te_r['bacc']:.4f}  ({elapsed/60:.1f} min)")

        all_results.append(row)
        done_pairs.add((str(subj), str(model_name)))

        # Salva CSV dopo ogni modello (resume-safe, sopravvive a crash)
        pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)

print(f"\n{'='*60}")
print(f"LOOP COMPLETATO — {len(all_subjects)} soggetti x {len(MODEL_NAMES)} modelli")
print(f"Risultati salvati in: {RESULTS_CSV}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# SUMMARY — tabella risultati aggregati
# ============================================================

df_ss = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(all_results)

# Media e std per modello (aggregato su tutti i soggetti)
summary = df_ss.groupby("model").agg(
    mean_test_bacc=("test_bacc", "mean"),
    std_test_bacc=("test_bacc", "std"),
    mean_val_bacc=("val_bacc", "mean"),
    mean_test_acc=("test_acc", "mean"),
    n_subjects=("subject", "count"),
).round(4)

print(f"\n=== EEG_08b Subject-Specific | {N_CLASSES} classi | Chance={chance_level:.1%} ===")
print(summary.to_string())
print(f"\nChance level: {chance_level:.4f}")

if len(summary) > 0:
    best_model = summary["mean_test_bacc"].idxmax()
    best_bacc  = summary["mean_test_bacc"].max()
    print(f"Modello migliore: {best_model} → {best_bacc:.4f} "
          f"({best_bacc/chance_level:.2f}x chance)")

In [ ]:
# ============================================================
# BOXPLOT — distribuzione per soggetto
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"EEG_08b — ChebGCN Subject-Specific | {N_CLASSES} classi ({CLUSTER_SCHEME}){NORM_TAG}",
    fontsize=13, fontweight="bold"
)

# ── Balanced Accuracy boxplot ──
ax = axes[0]
sns.boxplot(data=df_ss, x="model", y="test_bacc", ax=ax, palette="Set2")
ax.axhline(chance_level, color="red", linestyle="--", linewidth=1.5,
           label=f"Chance ({chance_level:.1%})")
ax.set_title("Balanced Accuracy (test)")
ax.set_ylabel("Balanced Accuracy")
ax.set_xlabel("Modello")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

# ── Accuracy boxplot ──
ax = axes[1]
sns.boxplot(data=df_ss, x="model", y="test_acc", ax=ax, palette="Set2")
ax.axhline(chance_level, color="red", linestyle="--", linewidth=1.5,
           label=f"Chance ({chance_level:.1%})")
ax.set_title("Accuracy (test)")
ax.set_ylabel("Accuracy")
ax.set_xlabel("Modello")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

plt.tight_layout()

fig_path = project_root / "figures" / f"eeg08b_ss_{N_CLASSES}{NORM_TAG}_boxplot.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: {fig_path}")